# 02 EDA

Cel notebooka:
- analiza rozk?ad?w,
- analiza klas sell/hold/buy,
- korelacje i wsp??liniowo?? cech.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

def find_project_root():
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / 'src').exists() and (p / 'data').exists():
            return p
    raise RuntimeError('Nie moge znalezc katalogu projektu.')

PROJECT_ROOT = find_project_root()
train_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "train.parquet")
val_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "val.parquet")
test_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "test.parquet")

full_df = pd.concat([train_df, val_df, test_df], axis=0)
print("train/val/test:", len(train_df), len(val_df), len(test_df))
print("full:", len(full_df))


In [ ]:
full_df.describe().T.head(15)


In [ ]:
class_counts = full_df["target"].value_counts().sort_index()
class_counts


In [ ]:
plt.figure(figsize=(6,4))
class_counts.plot(kind="bar")
plt.title("Class distribution (0=sell, 1=hold, 2=buy)")
plt.ylabel("count")
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
full_df["future_return"].hist(bins=80)
plt.title("Future return histogram")
plt.show()


In [ ]:
selected = [c for c in ["close_open_pct", "high_low_pct", "rsi_14", "macd_hist", "volume_zscore_24", "future_return"] if c in full_df.columns]
selected


In [ ]:
if selected:
    corr = full_df[selected].corr(numeric_only=True)
    plt.figure(figsize=(8,6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation heatmap")
    plt.show()


In [ ]:
numeric_cols = [c for c in train_df.select_dtypes(include=[np.number]).columns if c not in ["target", "future_return"]]
sample_cols = numeric_cols[:15]
X = train_df[sample_cols].replace([np.inf, -np.inf], np.nan).dropna()

vif = []
for i, col in enumerate(X.columns):
    vif.append((col, variance_inflation_factor(X.values, i)))

vif_df = pd.DataFrame(vif, columns=["feature", "VIF"]).sort_values("VIF", ascending=False)
vif_df.head(15)
